# Solution A1 · OLS with the normal equation and with the SVD

**Type**: Implement from scratch

## Objective

Implement OLS regression in two ways, with the normal equation and with the SVD decomposition, and find out at
what point the two methods stop giving the same result.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
X = rng.normal(size=(100, 3))
y = 3 + X @ np.array([2.0, -1.0, 0.5]) + rng.normal(scale=0.5, size=100)

x_poly = np.linspace(0, 1, 60)
y_poly = np.sin(2 * np.pi * x_poly) + rng.normal(scale=0.1, size=60)

## Commented solution

The normal equation is solved with `np.linalg.solve(AtA, Aty)`: it is more stable and faster than computing the inverse. The
SVD, instead, works directly on A: if A = U·Σ·Vᵀ, then the least squares solution is V·Σ⁺·Uᵀ·y, where
Σ⁺ holds the inverses of the singular values, and zero in place of the negligible ones.

In [ ]:
def _with_ones(X):
    return np.column_stack([np.ones(len(X)), X])


def ols_normal(X, y):
    A = _with_ones(X)
    beta = np.linalg.solve(A.T @ A, A.T @ y)
    return beta[0], beta[1:]


def ols_svd(X, y, rcond=1e-12):
    A = _with_ones(X)
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    S_inv = np.where(S > rcond * S.max(), 1 / S, 0.0)        # tiny singular values get zeroed
    beta = Vt.T @ (S_inv * (U.T @ y))
    return beta[0], beta[1:]

On the well-conditioned problem the two functions agree up to machine precision. The interesting comparison
is on the Vandermonde matrix, where the condition number grows fast with the degree.

In [ ]:
rows = []
for degree in [4, 6, 8, 10]:
    A = np.vander(x_poly, degree + 1, increasing=True)
    reference = np.linalg.lstsq(A, y_poly, rcond=None)[0]

    beta_n = np.r_[ols_normal(A[:, 1:], y_poly)[0], ols_normal(A[:, 1:], y_poly)[1]]
    beta_s = np.r_[ols_svd(A[:, 1:], y_poly)[0], ols_svd(A[:, 1:], y_poly)[1]]

    rows.append({
        "degree": degree,
        "cond_A": np.linalg.cond(A),
        "cond_AtA": np.linalg.cond(A.T @ A),
        "err_normal": np.abs(beta_n - reference).max(),
        "err_svd": np.abs(beta_s - reference).max(),
    })

table = pd.DataFrame(rows)
pd.options.display.float_format = "{:.3g}".format
table

In [ ]:
critical_degree = int(table.loc[table.err_normal > 1e-3, "degree"].min())
critical_degree

## Check

In [ ]:
with verify():
    ref = LinearRegression().fit(X, y)
    i_n, c_n = ols_normal(X, y)
    i_s, c_s = ols_svd(X, y)
    check(close(i_n, ref.intercept_) and close(c_n, ref.coef_),
          "ols_normal matches LinearRegression",
          "ols_normal does not match LinearRegression: check the column of ones and the normal equation")
    check(close(i_s, ref.intercept_) and close(c_s, ref.coef_),
          "ols_svd matches LinearRegression",
          "ols_svd does not match LinearRegression: check 1/S and the reconstruction of beta")

    check(list(table.columns) == ["degree", "cond_A", "cond_AtA", "err_normal", "err_svd"],
          "the table has the required columns", "the columns of `table` are not the required ones")
    check((table.err_svd < 1e-6).all(),
          "the SVD stays accurate at every degree", "err_svd should stay below 1e-6 at all degrees")
    check(table.err_normal.is_monotonic_increasing,
          "the error of the normal equation grows with the degree", "err_normal should grow with the degree")
    ratio = np.log10(table.cond_AtA) / np.log10(table.cond_A)
    check(np.allclose(ratio, 2, atol=0.25),
          "cond(AtA) is about cond(A) squared",
          "cond_AtA should be about the square of cond_A: did you compute np.linalg.cond(A.T @ A)?")
    check(critical_degree == 8, "correct critical degree",
          "look at which row of `table` has err_normal exceeding 1e-3 for the first time")
done()

## What you learned

- `cond(AᵀA) = cond(A)²`: squaring the matrix halves the available significant digits. At degree 10 the
  condition number of A is of the order of 10⁷, that of AᵀA of 10¹⁴, and with 16 digits in double precision
  barely two are left: the error of the normal equation is of the order of 10¹.
- The SVD does not go through AᵀA and keeps an error close to machine precision even where the normal equation is
  useless. That is why `LinearRegression` uses `scipy.linalg.lstsq`.
- "Correct formula" and "correct algorithm" are not the same thing: in finite arithmetic how you compute matters.

## Common pitfall

The conditioning problem does not concern only polynomials: it shows up whenever two or more features are nearly
collinear or have very different scales. With the normal equation it happens silently: no error, just wrong
coefficients. Scaling the features (and not using `np.linalg.inv`) reduces the risk but does not eliminate it.

## Interview question

> *Click on a question to show its answer.*

<details><summary>A colleague computes the regression with (XᵀX)⁻¹Xᵀy and sometimes gets absurd coefficients. Where do you start?</summary>

I check `np.linalg.cond(X.T @ X)`: if it is close to 10¹⁶ the matrix is numerically singular, typically because of collinear
features or very different scales. Then I switch to a solver based on QR or SVD (`np.linalg.lstsq`,
`LinearRegression`), which does not square the conditioning, I scale the features and, if the collinearity is
real, I consider a Ridge.

</details>